<a href="https://www.kaggle.com/code/mobeenfatimah/name-that-llm-ensemble?scriptVersionId=366622346" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Name That LLM — Text Fingerprinting Ensemble

Offline CPU-friendly ensemble for the 18-class authorship task:
- word TF-IDF + LinearSVC
- character TF-IDF + LinearSVC
- character-boundary TF-IDF + LinearSVC
- normalized decision-score averaging

**Validation:** GroupKFold by `prompt_id`. Prompt IDs are used only to form validation folds, never as prediction features. This avoids the prompt leakage that a random row split could create.

## 1. Libraries Imports and configuration

In [1]:
# 1. Imports and configuration
import os, gc, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score, classification_report
warnings.filterwarnings("ignore")

TRAIN_PATH = "/kaggle/input/competitions/name-that-llm/train.csv"
TEST_PATH = "/kaggle/input/competitions/name-that-llm/test.csv"
RUN_CV = True
N_SPLITS = 5
SEED = 42

LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",
    "gpt-6-astra", "gemini-3.1-pro-preview", "gemini-3.6-flash",
    "gemini-3.7-flash", "gemini-3.8-flash", "claude-fable-5",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5",
    "claude-sonnet-5", "qwen3.8-max", "glm-5.3", "glm-5.3-flash",
    "kimi-k3"
]


## 2. Loads and Inspect data 

In [2]:
# 2. Load and inspect data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
assert {"id", "prompt_id", "text", "model"}.issubset(train.columns)
assert {"id", "text"}.issubset(test.columns)

train["text"] = train["text"].fillna("").astype(str)
test["text"] = test["text"].fillna("").astype(str)
train["model"] = train["model"].astype(str)

unknown = sorted(set(train["model"]) - set(LABELS))
assert not unknown, f"Unexpected labels: {unknown}"

print("Train:", train.shape, "| Test:", test.shape)
print("Classes:", train["model"].nunique())
print("Unique prompt IDs:", train["prompt_id"].nunique())
print("Duplicate training texts:", train["text"].duplicated().sum())
display(train["model"].value_counts().reindex(LABELS))


Train: (2699, 4) | Test: (3, 2)
Classes: 18
Unique prompt IDs: 150
Duplicate training texts: 0


model
gpt-5.5                   150
gpt-5.6-luna              150
gpt-5.6-sol               150
gpt-5.6-terra             150
gpt-6-astra               150
gemini-3.1-pro-preview    150
gemini-3.6-flash          150
gemini-3.7-flash          150
gemini-3.8-flash          150
claude-fable-5            149
claude-opus-4-6           150
claude-opus-4-8           150
claude-opus-5             150
claude-sonnet-5           150
qwen3.8-max               150
glm-5.3                   150
glm-5.3-flash             150
kimi-k3                   150
Name: count, dtype: int64

## 3. Feature engineering

Word n-grams capture vocabulary and short phrases. Character n-grams capture punctuation, spelling, morphology and formatting. `char_wb` captures character patterns within word boundaries. Each classifier contributes a normalized decision-score matrix to the ensemble.

In [3]:
# 4. Vectorizers and ensemble helper
def make_vectorizers():
    return {
        "word": TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.98,
            max_features=120_000, sublinear_tf=True,
            strip_accents="unicode", dtype=np.float32
        ),
        "char": TfidfVectorizer(
            analyzer="char", ngram_range=(3, 5), min_df=2,
            max_features=250_000, sublinear_tf=True, dtype=np.float32
        ),
        "char_wb": TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=2,
            max_features=200_000, sublinear_tf=True, dtype=np.float32
        )
    }

def normalize_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    mu = scores.mean(axis=1, keepdims=True)
    sigma = scores.std(axis=1, keepdims=True)
    return (scores - mu) / np.maximum(sigma, 1e-8)

def fit_ensemble_predict(X_fit, y_fit, X_pred, C=1.0):
    vectorizers = make_vectorizers()
    score_list = []
    fitted = {}

    for name, vectorizer in vectorizers.items():
        A = vectorizer.fit_transform(X_fit)
        B = vectorizer.transform(X_pred)
        clf = LinearSVC(C=C, class_weight="balanced", random_state=SEED)
        clf.fit(A, y_fit)
        score_list.append(normalize_scores(clf.decision_function(B)))
        fitted[name] = (vectorizer, clf)
        print(name, "features:", A.shape[1])
        del A, B
        gc.collect()

    avg_scores = np.mean(np.stack(score_list, axis=0), axis=0)
    classes = fitted["word"][1].classes_
    pred = classes[np.argmax(avg_scores, axis=1)]
    return pred, avg_scores, fitted


## 4. Grouped cross-validation

The competition states that prompts in the test set are unseen. Therefore, hold out complete prompt groups. This is a stricter and more realistic estimate than a random row split. The target metric is balanced accuracy.

In [4]:
# 5. Out-of-fold validation
X = train["text"].reset_index(drop=True)
y = train["model"].reset_index(drop=True)
groups = train["prompt_id"].reset_index(drop=True)

if RUN_CV:
    splitter = GroupKFold(n_splits=N_SPLITS)
    oof_pred = np.empty(len(train), dtype=object)

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups), 1):
        print(f"\\n========== Fold {fold}/{N_SPLITS} ==========")
        pred, _, _ = fit_ensemble_predict(
            X.iloc[tr_idx].tolist(),
            y.iloc[tr_idx].to_numpy(),
            X.iloc[va_idx].tolist(),
            C=1.0
        )
        oof_pred[va_idx] = pred
        print("Fold balanced accuracy:",
              round(balanced_accuracy_score(y.iloc[va_idx], pred), 5))
        gc.collect()

    print("\\nOverall grouped OOF balanced accuracy:",
          round(balanced_accuracy_score(y, oof_pred), 5))
    print(classification_report(y, oof_pred, labels=LABELS, zero_division=0))
else:
    print("CV skipped. Set RUN_CV=True to evaluate.")


\n========== Fold 1/5 ==========
word features: 40149
char features: 110598
char_wb features: 57434
Fold balanced accuracy: 0.29444
\n========== Fold 2/5 ==========
word features: 39933
char features: 111161
char_wb features: 57859
Fold balanced accuracy: 0.29074
\n========== Fold 3/5 ==========
word features: 40207
char features: 110417
char_wb features: 57318
Fold balanced accuracy: 0.29074
\n========== Fold 4/5 ==========
word features: 40228
char features: 109566
char_wb features: 56850
Fold balanced accuracy: 0.32222
\n========== Fold 5/5 ==========
word features: 40219
char features: 111001
char_wb features: 57771
Fold balanced accuracy: 0.29834
\nOverall grouped OOF balanced accuracy: 0.29928
                        precision    recall  f1-score   support

               gpt-5.5       0.21      0.21      0.21       150
          gpt-5.6-luna       0.18      0.23      0.20       150
           gpt-5.6-sol       0.24      0.15      0.19       150
         gpt-5.6-terra       0.25 

## 5. Fit all training data and predict test

After grouped validation, refit all three models on every labeled row and predict the test passages.

In [5]:
# 6. Final training and prediction
test_pred, test_scores, final_models = fit_ensemble_predict(
    train["text"].tolist(),
    train["model"].to_numpy(),
    test["text"].tolist(),
    C=1.0
)

submission = pd.DataFrame({
    "id": test["id"].values,
    "model": test_pred
})

assert list(submission.columns) == ["id", "model"]
assert len(submission) == len(test)
assert submission["id"].notna().all()
assert submission["id"].is_unique
assert submission["model"].notna().all()
assert set(submission["model"]).issubset(set(LABELS))

display(submission.head())
display(submission["model"].value_counts())


word features: 48823
char features: 120718
char_wb features: 62596


,id,model
0,00353fb1c541,gemini-3.6-flash
1,005f5a9a6139,gpt-5.6-terra
2,0062a53d5487,gemini-3.7-flash


model
gemini-3.6-flash    1
gpt-5.6-terra       1
gemini-3.7-flash    1
Name: count, dtype: int64

## 6. Save the required Kaggle artifact

In [6]:
# 7. Save the required Kaggle artifact
OUTPUT_PATH = "submission.parquet"
submission.to_parquet(OUTPUT_PATH, index=False)

# Verify the written file
check = pd.read_parquet(OUTPUT_PATH)
assert check.equals(submission.reset_index(drop=True))
print("Saved and verified:", OUTPUT_PATH)
print("Rows:", len(check), "| Columns:", check.columns.tolist())


Saved and verified: submission.parquet
Rows: 3 | Columns: ['id', 'model']


## 7. Tuning ideas

Use grouped OOF balanced accuracy to compare experiments:
1. Try LinearSVC `C` values of 0.5, 1.0 and 2.0.
2. Try character n-grams `(2, 5)` or `(3, 6)`.
3. Adjust ensemble weights only if grouped validation consistently supports the change.
4. Review per-class recall to see which models are confused.

Do not tune against a random split: it can overstate generalization. Keep internet disabled and verify the final notebook completes within the competition's 9-hour limit. No approach can guarantee the maximum leaderboard score; this is a strong, reproducible baseline designed around the data structure you provided.